# Comparative Analysis of Machine Learning Models for Money Demand Forecasting — M3 (Broad Money)

This notebook implements the M3 (broad money demand) forecasting analysis from:

> Sikhwal S., Sen S. *Comparative Analysis of Machine Learning Models for Money Demand Forecasting in the Indian Economy.* HSE Economic Journal. 2024; 28(1): 133-158.

**Dataset:** Monthly time series, India, 1997-2021. Source: CEIC.  
**Target variable:** M3_real = M3SA / CPISA (real broad money balances, log-differenced)  
**Features:** IPISA, Govt securities yield, NEER, BSE market capitalisation  
**Models:** AR(1) benchmark, Random Forest, Gradient Boosting, XGBoost, SVR, LASSO, LSTM  
**Validation:** Expanding window cross-validation with K = 2 to 7 folds  
**Evaluation:** MSE, RMSE, MAPE, SMAPE, TIC; Diebold-Mariano test with Harvey adjustment  

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.ar_model import AutoReg
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from helper_functions import *

## 2. Data Loading and Preparation

### 2.1 Read the Excel file and select M3 variables

Variables selected as per the log-linearised Money Demand Function (Equation 1 in the paper):
- **M3SA** — seasonally adjusted nominal broad money aggregate
- **CPISA** — seasonally adjusted CPI (price level, used to deflate M3SA to real balances)
- **IPISA** — Index of Industrial Production (income proxy, seasonally adjusted)
- **Govt securities yield** — interest rate variable for M3 (not log-transformed; in percentage form)
- **NEER** — Nominal Effective Exchange Rate
- **BSE mkt cap mn** — Bombay Stock Exchange market capitalisation (financial stability proxy)

In [2]:
df = pd.read_excel("DataMoneyDemand.xlsx")
df = df[['Date', 'M3SA', 'CPISA', 'IPISA', 'Govt securities yield', 'NEER', 'BSE mkt cap mn']]

df["Date"] = pd.to_datetime(df["Date"], format='%Y-%m-%d')
df = df.dropna()
df.set_index('Date', inplace=True)
df.head()

,M3SA,CPISA,IPISA,Govt securities yield,NEER,BSE mkt cap mn
Date,,,,,,
1997-01-01,6.749799e+06,43.322394,51.012669,13.6600,125.76,4582610.0
1997-02-01,6.823220e+06,43.616447,51.179397,13.4275,128.69,4846240.0
1997-03-01,6.947731e+06,43.906066,47.080132,13.4328,129.66,4639150.0
1997-04-01,7.005230e+06,44.082589,56.884312,12.8648,130.39,5020820.0
1997-05-01,7.103107e+06,43.838752,50.131279,12.8741,129.56,5063910.0


### 2.2 Compute real money balances and apply log transformation

Following Equation 1 in the paper, real money balances are computed by dividing the seasonally
adjusted nominal money stock (M3SA) by the seasonally adjusted price level (CPISA).
Natural log is then applied to all variables **except** the interest rate (Govt securities yield),
which is already in percentage form.

In [3]:
# Compute real money balances: M3_real = M3SA / CPISA (Equation 1 in the paper)
df['M3_real'] = df['M3SA'] / df['CPISA']

# Drop the nominal M3SA and CPISA columns — no longer needed
df = df.drop(columns=['M3SA', 'CPISA'])

# Apply log to all variables except Govt securities yield (interest rate, in percentage form)
df[["M3_real", "IPISA", "NEER", "BSE mkt cap mn"]] =     df[["M3_real", "IPISA", "NEER", "BSE mkt cap mn"]].apply(np.log)
df.head()

,IPISA,Govt securities yield,NEER,BSE mkt cap mn,M3_real
Date,,,,,
1997-01-01,3.932074,13.6600,4.834375,15.337779,11.956354
1997-02-01,3.935337,13.4275,4.857406,15.393714,11.960408
1997-03-01,3.851851,13.4328,4.864916,15.350042,11.971873
1997-04-01,4.041020,12.8648,4.870530,15.429104,11.976103
1997-05-01,3.914645,12.8741,4.864144,15.437649,11.995525


## 3. Stationarity Tests

### 3.1 Unit root tests at level (ADF and Phillips-Perron)

The null hypothesis for both ADF and PP tests is that the series has a unit root (non-stationary).
Maximum lag length: 15

In [4]:
print("=== Unit Root Tests: Level ===")
adf_test(df)

=== Unit Root Tests: Level ===
IPISA:
  ADF Statistic : -1.8106
  p-value       : 0.3752
  Critical Value (1%): -3.4527
  Critical Value (5%): -2.8714
  Critical Value (10%): -2.5720
  Phillips-Perron:      Phillips-Perron Test (Z-tau)    
Test Statistic                 -2.067
P-value                         0.258
Lags                               16
-------------------------------------

Trend: Constant
Critical Values: -3.45 (1%), -2.87 (5%), -2.57 (10%)
Null Hypothesis: The process contains a unit root.
Alternative Hypothesis: The process is weakly stationary.

Govt securities yield:
  ADF Statistic : -2.4763
  p-value       : 0.1214
  Critical Value (1%): -3.4528
  Critical Value (5%): -2.8714
  Critical Value (10%): -2.5720
  Phillips-Perron:      Phillips-Perron Test (Z-tau)    
Test Statistic                 -2.829
P-value                         0.054
Lags                               16
-------------------------------------

Trend: Constant
Critical Values: -3.45 (1%), -2.87

### 3.2 First differencing and re-testing

All series are non-stationary at level. We apply first differencing to obtain I(1) stationarity.

In [5]:
differenced_data = df.diff(axis=0).dropna()
assert differenced_data.shape == (len(df) - 1, df.shape[1])

print("=== Unit Root Tests: First Difference ===")
adf_test(differenced_data)

=== Unit Root Tests: First Difference ===
IPISA:
  ADF Statistic : -14.1495
  p-value       : 0.0000
  Critical Value (1%): -3.4527
  Critical Value (5%): -2.8714
  Critical Value (10%): -2.5720
  Phillips-Perron:      Phillips-Perron Test (Z-tau)    
Test Statistic                -35.215
P-value                         0.000
Lags                               16
-------------------------------------

Trend: Constant
Critical Values: -3.45 (1%), -2.87 (5%), -2.57 (10%)
Null Hypothesis: The process contains a unit root.
Alternative Hypothesis: The process is weakly stationary.

Govt securities yield:
  ADF Statistic : -8.1238
  p-value       : 0.0000
  Critical Value (1%): -3.4528
  Critical Value (5%): -2.8714
  Critical Value (10%): -2.5720
  Phillips-Perron:      Phillips-Perron Test (Z-tau)    
Test Statistic                -18.120
P-value                         0.000
Lags                               16
-------------------------------------

Trend: Constant
Critical Values: -3.45

## 4. Data Preparation for Modelling

### 4.1 Construct the M3 modelling dataframe and apply log-differencing

In [6]:
# Rename M3_real to y3 for modelling
M3_df = df.assign(y3=df["M3_real"]).drop("M3_real", axis=1)

# Store the 2018-01-01 level value of M3_real for inverse transformation later
temp = df.loc[pd.to_datetime('2018-01-01')]["M3_real"]

# Apply first difference (log-difference = continuous growth rate)
M3_df = M3_df.diff().dropna()
M3_df.head()

,IPISA,Govt securities yield,NEER,BSE mkt cap mn,y3
Date,,,,,
1997-02-01,0.003263,-0.2325,0.023031,0.055934,0.004054
1997-03-01,-0.083486,0.0053,0.007509,-0.043672,0.011465
1997-04-01,0.189169,-0.5680,0.005614,0.079062,0.004230
1997-05-01,-0.126374,0.0093,-0.006386,0.008546,0.019422
1997-06-01,0.014460,-0.2883,0.000386,0.150261,0.010823


### 4.2 Train/test split

The dataset is split at January 2018: ~80% training (1997-2017), ~20% testing (2018-2021), yielding N=48 test observations.

In [7]:
X = M3_df[["BSE mkt cap mn", "IPISA", "Govt securities yield", "NEER"]].to_numpy()
y = np.array(M3_df['y3'])

X, X_test, y, y_test, idx = make_split(M3_df, X, y, '2018-01-01')
print(f"X_train shape : {X.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"y_train shape : {y.shape}")
print(f"y_test shape  : {y_test.shape}")

X_train shape : (251, 4)
X_test shape  : (48, 4)
y_train shape : (251,)
y_test shape  : (48,)


## 5. Model Training and Evaluation

### 5.1 Helper: evaluate_model_for_split

Trains a given sklearn-compatible model using expanding window cross-validation across K = 2 to 7 folds, predicts on the test set, reverses the log-difference transformation, and prints evaluation metrics.

In [8]:
def evaluate_model_for_split(splits, train_X, train_Y, test_X, test_Y, model_name, *args, **kwargs):
    """Train model across expanding window folds and evaluate on test set.

    A fresh model instance is created for each split K to avoid accumulation
    of training state across folds. Predictions are reverse-transformed from
    log-differences back to levels using the 2018-01-01 anchor value (temp).

    Returns:
        y_preds: list of untransformed predictions, one per split
    """
    y_preds = []
    print(model_name)
    for split in splits:
        model = model_name(*args, **kwargs)
        _, model = expanding_window_cv_with_splits(train_X, train_Y, model, num_splits=split)
        y_pred_diff = model.predict(test_X)

        # Reverse log-difference transformation
        y_pred_untransformed = np.cumsum(y_pred_diff[1:]) + temp
        y_test_untransformed = np.cumsum(test_Y[1:]) + temp

        mean_rmse, mean_mse, mean_mape, mean_smape, mean_tic = metrics(
            y_test_untransformed, y_pred_untransformed)
        print_metrics(mean_rmse, mean_mse, mean_mape, mean_smape, mean_tic, split)
        y_preds.append(y_pred_untransformed)

    return y_preds

### 5.2 AR(1) — Benchmark Model

In [9]:
splits = [2, 3, 4, 5, 6, 7]

for split in splits:
    result = AR_expanding_window_cv_with_splits(X, y, n_splits=split)
    mean_rmse, mean_mse, mean_mape, mean_smape, mean_tic = [], [], [], [], []
    print(f"K = {split}")

    for (X_train, y_train, X_val, y_val) in result:
        AR = AutoReg(y_train, lags=1).fit()
        y_pred_ar = AR.predict(start=len(y_val), end=len(y_val) + len(y_test) - 1)

        mean_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred_ar)))
        mean_mse.append(mean_squared_error(y_test, y_pred_ar))
        mean_mape.append(mean_absolute_percentage_error(y_test, y_pred_ar))
        mean_smape.append(symmetric_mean_absolute_percentage_error(y_test, y_pred_ar))
        mean_tic.append(theil_inequality_coeff(y_test, y_pred_ar))

        y_pred_untransformed_ar = np.cumsum(y_pred_ar[1:]) + temp
        y_test_untransformed = np.cumsum(y_test[1:]) + temp

    mean_rmse, mean_mse, mean_mape, mean_smape, mean_tic = metrics(
        y_test_untransformed, y_pred_untransformed_ar)
    print_metrics(mean_rmse, mean_mse, mean_mape, mean_smape, mean_tic, split)

K = 2
Test Set for K = 2
Mean of MSE   : 0.00259
Mean of RMSE  : 0.05092
Mean of MAPE  : 0.29057
Mean of SMAPE : 0.28989
Mean of TIC   : 0.00185
 
K = 3
Test Set for K = 3
Mean of MSE   : 0.00266
Mean of RMSE  : 0.05159
Mean of MAPE  : 0.29489
Mean of SMAPE : 0.29419
Mean of TIC   : 0.00187
 
K = 4
Test Set for K = 4
Mean of MSE   : 0.00289
Mean of RMSE  : 0.05371
Mean of MAPE  : 0.30752
Mean of SMAPE : 0.30677
Mean of TIC   : 0.00195
 
K = 5
Test Set for K = 5
Mean of MSE   : 0.00259
Mean of RMSE  : 0.05092
Mean of MAPE  : 0.29057
Mean of SMAPE : 0.28989
Mean of TIC   : 0.00185
 
K = 6
Test Set for K = 6
Mean of MSE   : 0.00280
Mean of RMSE  : 0.05288
Mean of MAPE  : 0.30231
Mean of SMAPE : 0.30158
Mean of TIC   : 0.00192
 
K = 7
Test Set for K = 7
Mean of MSE   : 0.00290
Mean of RMSE  : 0.05384
Mean of MAPE  : 0.30819
Mean of SMAPE : 0.30743
Mean of TIC   : 0.00196
 


### 5.3 Random Forest Regression

In [10]:
splits = [2, 3, 4, 5, 6, 7]

y_pred_random_forest = evaluate_model_for_split(
    splits, X, y, X_test, y_test, RandomForestRegressor)

<class 'sklearn.ensemble._forest.RandomForestRegressor'>
Test Set for K = 2
Mean of MSE   : 0.00162
Mean of RMSE  : 0.04023
Mean of MAPE  : 0.22299
Mean of SMAPE : 0.22258
Mean of TIC   : 0.00146
 
Test Set for K = 3
Mean of MSE   : 0.00188
Mean of RMSE  : 0.04337
Mean of MAPE  : 0.23975
Mean of SMAPE : 0.23927
Mean of TIC   : 0.00158
 
Test Set for K = 4
Mean of MSE   : 0.00159
Mean of RMSE  : 0.03992
Mean of MAPE  : 0.21919
Mean of SMAPE : 0.21879
Mean of TIC   : 0.00145
 
Test Set for K = 5
Mean of MSE   : 0.00163
Mean of RMSE  : 0.04033
Mean of MAPE  : 0.22302
Mean of SMAPE : 0.22261
Mean of TIC   : 0.00147
 
Test Set for K = 6
Mean of MSE   : 0.00183
Mean of RMSE  : 0.04273
Mean of MAPE  : 0.23700
Mean of SMAPE : 0.23653
Mean of TIC   : 0.00155
 
Test Set for K = 7
Mean of MSE   : 0.00167
Mean of RMSE  : 0.04082
Mean of MAPE  : 0.22167
Mean of SMAPE : 0.22125
Mean of TIC   : 0.00148
 


### 5.4 Gradient Boosting

In [19]:
splits = [2, 3, 4, 5, 6, 7]

y_pred_gradboost = evaluate_model_for_split(
    splits, X, y, X_test, y_test, GradientBoostingRegressor)

<class 'sklearn.ensemble._gb.GradientBoostingRegressor'>
Test Set for K = 2
Mean of MSE   : 0.00220
Mean of RMSE  : 0.04690
Mean of MAPE  : 0.26382
Mean of SMAPE : 0.26324
Mean of TIC   : 0.00170
 
Test Set for K = 3
Mean of MSE   : 0.00161
Mean of RMSE  : 0.04017
Mean of MAPE  : 0.22493
Mean of SMAPE : 0.22452
Mean of TIC   : 0.00146
 
Test Set for K = 4
Mean of MSE   : 0.00224
Mean of RMSE  : 0.04731
Mean of MAPE  : 0.25877
Mean of SMAPE : 0.25819
Mean of TIC   : 0.00172
 
Test Set for K = 5
Mean of MSE   : 0.00222
Mean of RMSE  : 0.04717
Mean of MAPE  : 0.26592
Mean of SMAPE : 0.26534
Mean of TIC   : 0.00171
 
Test Set for K = 6
Mean of MSE   : 0.00240
Mean of RMSE  : 0.04900
Mean of MAPE  : 0.27832
Mean of SMAPE : 0.27769
Mean of TIC   : 0.00178
 
Test Set for K = 7
Mean of MSE   : 0.00200
Mean of RMSE  : 0.04469
Mean of MAPE  : 0.24876
Mean of SMAPE : 0.24825
Mean of TIC   : 0.00162
 


### 5.5 XGBoost

In [12]:
splits = [2, 3, 4, 5, 6, 7]

y_pred_xgboost = evaluate_model_for_split(
    splits, X, y, X_test, y_test, XGBRegressor)

<class 'xgboost.sklearn.XGBRegressor'>
Test Set for K = 2
Mean of MSE   : 0.00322
Mean of RMSE  : 0.05677
Mean of MAPE  : 0.33955
Mean of SMAPE : 0.33870
Mean of TIC   : 0.00206
 
Test Set for K = 3
Mean of MSE   : 0.00380
Mean of RMSE  : 0.06167
Mean of MAPE  : 0.37238
Mean of SMAPE : 0.37138
Mean of TIC   : 0.00224
 
Test Set for K = 4
Mean of MSE   : 0.00369
Mean of RMSE  : 0.06079
Mean of MAPE  : 0.35960
Mean of SMAPE : 0.35863
Mean of TIC   : 0.00221
 
Test Set for K = 5
Mean of MSE   : 0.00322
Mean of RMSE  : 0.05677
Mean of MAPE  : 0.33955
Mean of SMAPE : 0.33870
Mean of TIC   : 0.00206
 
Test Set for K = 6
Mean of MSE   : 0.00486
Mean of RMSE  : 0.06970
Mean of MAPE  : 0.40072
Mean of SMAPE : 0.39945
Mean of TIC   : 0.00253
 
Test Set for K = 7
Mean of MSE   : 0.00473
Mean of RMSE  : 0.06876
Mean of MAPE  : 0.41805
Mean of SMAPE : 0.41681
Mean of TIC   : 0.00250
 


### 5.6 Support Vector Regression (SVR)

In [13]:
splits = [2, 3, 4, 5, 6, 7]

y_pred_svr = evaluate_model_for_split(
    splits, X, y, X_test, y_test, SVR)

<class 'sklearn.svm._classes.SVR'>
Test Set for K = 2
Mean of MSE   : 0.00480
Mean of RMSE  : 0.06929
Mean of MAPE  : 0.40566
Mean of SMAPE : 0.40440
Mean of TIC   : 0.00252
 
Test Set for K = 3
Mean of MSE   : 0.00480
Mean of RMSE  : 0.06929
Mean of MAPE  : 0.40566
Mean of SMAPE : 0.40440
Mean of TIC   : 0.00252
 
Test Set for K = 4
Mean of MSE   : 0.00480
Mean of RMSE  : 0.06929
Mean of MAPE  : 0.40566
Mean of SMAPE : 0.40440
Mean of TIC   : 0.00252
 
Test Set for K = 5
Mean of MSE   : 0.00480
Mean of RMSE  : 0.06929
Mean of MAPE  : 0.40566
Mean of SMAPE : 0.40440
Mean of TIC   : 0.00252
 
Test Set for K = 6
Mean of MSE   : 0.00480
Mean of RMSE  : 0.06929
Mean of MAPE  : 0.40566
Mean of SMAPE : 0.40440
Mean of TIC   : 0.00252
 
Test Set for K = 7
Mean of MSE   : 0.00480
Mean of RMSE  : 0.06929
Mean of MAPE  : 0.40566
Mean of SMAPE : 0.40440
Mean of TIC   : 0.00252
 


### 5.7 LASSO Regression

In [14]:
splits = [2, 3, 4, 5, 6, 7]

y_pred_lasso = evaluate_model_for_split(
    splits, X, y, X_test, y_test, Lasso,
    fit_intercept=False, max_iter=20000, tol=1e-2, precompute=True, alpha=1)

<class 'sklearn.linear_model._coordinate_descent.Lasso'>
Test Set for K = 2
Mean of MSE   : 0.01778
Mean of RMSE  : 0.13333
Mean of MAPE  : 0.86918
Mean of SMAPE : 0.87388
Mean of TIC   : 0.00487
 
Test Set for K = 3
Mean of MSE   : 0.01778
Mean of RMSE  : 0.13333
Mean of MAPE  : 0.86918
Mean of SMAPE : 0.87388
Mean of TIC   : 0.00487
 
Test Set for K = 4
Mean of MSE   : 0.01778
Mean of RMSE  : 0.13333
Mean of MAPE  : 0.86918
Mean of SMAPE : 0.87388
Mean of TIC   : 0.00487
 
Test Set for K = 5
Mean of MSE   : 0.01778
Mean of RMSE  : 0.13333
Mean of MAPE  : 0.86918
Mean of SMAPE : 0.87388
Mean of TIC   : 0.00487
 
Test Set for K = 6
Mean of MSE   : 0.01778
Mean of RMSE  : 0.13333
Mean of MAPE  : 0.86918
Mean of SMAPE : 0.87388
Mean of TIC   : 0.00487
 
Test Set for K = 7
Mean of MSE   : 0.01778
Mean of RMSE  : 0.13333
Mean of MAPE  : 0.86918
Mean of SMAPE : 0.87388
Mean of TIC   : 0.00487
 


### 5.8 LSTM (Long Short-Term Memory)

#### 5.8.1 Device setup, model definition, and test tensor preparation

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Prepare test tensors for LSTM evaluation
X_test_tensor = torch.from_numpy(X_test).float().to(device).unsqueeze(0)
y_test_tensor = torch.from_numpy(y_test).float().to(device).unsqueeze(0)


class LSTMModel(nn.Module):
    """LSTM model for time series forecasting."""
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = dropout
        if self.num_layers == 1:
            self.lstm = nn.LSTM(input_size, hidden_size, 1, batch_first=True)
        else:
            self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                                batch_first=True, dropout=self.dropout)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out)
        return out.squeeze(-1)


def accuracy_measures(y_test, predictions_test):
    """Print evaluation metrics for LSTM predictions (accepts tensors)."""
    y_test = y_test.cpu().numpy()
    predictions_test = predictions_test.cpu().numpy()
    mse = mean_squared_error(y_test, predictions_test)
    print("RMSE :", np.sqrt(mse))
    print("MSE  :", mse)
    print("MAPE :", mean_absolute_percentage_error(y_test, predictions_test))
    print("SMAPE:", symmetric_mean_absolute_percentage_error(y_test, predictions_test))
    print("TIC  :", theil_inequality_coeff(y_test, predictions_test))
    print("")

#### 5.8.2 LSTM expanding window cross-validation function

In [16]:
def tf_expanding_window_cv_with_splits(X, y, num_epochs, criterion, n_splits=5,
                                       input_size=4, hidden_size=64, num_layers=3,
                                       dropout=0.2, learning_rate=0.1, device='cpu'):
    """Train LSTM with expanding window cross-validation using TimeSeriesSplit."""
    tsv = TimeSeriesSplit(n_splits=n_splits)
    batch_size = 32

    for train_idx, val_idx in tsv.split(X):
        X_train_fold = torch.from_numpy(X[train_idx]).float().to(device).unsqueeze(0)
        y_train_fold = torch.from_numpy(y[train_idx]).float().to(device).unsqueeze(0)
        X_val_fold   = torch.from_numpy(X[val_idx]).float().to(device).unsqueeze(0)
        y_val_fold   = torch.from_numpy(y[val_idx]).float().to(device).unsqueeze(0)

        lstm_model = LSTMModel(input_size, hidden_size, num_layers, 1, dropout).to(device)
        optimizer = torch.optim.SGD(lstm_model.parameters(), lr=learning_rate)

        train_loader = DataLoader(
            TensorDataset(X_train_fold, y_train_fold),
            batch_size=batch_size, shuffle=True)

        lstm_model.train()
        for epoch in range(num_epochs):
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = lstm_model(inputs)
                loss = criterion(outputs, labels)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        lstm_model.eval()
        with torch.no_grad():
            predictions = lstm_model(X_val_fold)
        val_loss = criterion(y_val_fold, predictions)

    return val_loss, lstm_model

#### 5.8.3 Hyperparameter grid search and test set evaluation

Exhaustive grid search over hidden size, number of layers, dropout rate, and learning rate for each fold K = 2 to 7. Best hyperparameters selected by lowest validation MSE. LSTM trained for **300 epochs**.

In [17]:
num_splits = [2, 3, 4, 5, 6, 7]
criterion  = nn.MSELoss()
best_hyperparameters = []

for split in num_splits:
    best_val_loss_for_split = np.inf
    best_hyperparams_for_split = {}

    for hidden_size in [32, 64, 128]:
        for num_layers in [1, 2, 3]:
            for dropout_rate in [0.0, 0.1, 0.2]:
                for learning_rate in [0.0001, 0.0005, 0.0008, 0.003]:
                    val_loss, model = tf_expanding_window_cv_with_splits(
                        X, y, 300, criterion,
                        n_splits=split, input_size=4,
                        hidden_size=hidden_size, num_layers=num_layers,
                        dropout=dropout_rate, learning_rate=learning_rate,
                        device=device)

                    if val_loss < best_val_loss_for_split:
                        best_val_loss_for_split = val_loss
                        best_hyperparams_for_split = {
                            'hidden_size': hidden_size,
                            'num_layers': num_layers,
                            'dropout_rate': dropout_rate,
                            'learning_rate': learning_rate,
                            'split_size': split}

    best_hyperparameters.append(best_hyperparams_for_split)

    print(f"Split K = {split} | Best params: {best_hyperparams_for_split}")

    model.eval()
    with torch.no_grad():
        predictions_test = model(X_test_tensor)

    y_pred_untransformed = np.cumsum(predictions_test.flatten().cpu().numpy()[1:]) + temp
    y_test_untransformed_lstm = np.cumsum(y_test_tensor.flatten().cpu().numpy()[1:]) + temp

    y_pred_untransformed_t = torch.tensor(y_pred_untransformed)
    y_test_untransformed_t = torch.tensor(y_test_untransformed_lstm)

    print(f"Test loss (MSE on levels): {criterion(y_test_untransformed_t, y_pred_untransformed_t).item():.5f}")
    accuracy_measures(y_test_untransformed_t, y_pred_untransformed_t)

print("\nBest hyperparameters per fold:")
for h in best_hyperparameters:
    print(h)

# Capture K=6 predictions for DM test
    if split == 6:
        y_pred_lstm = y_pred_untransformed
        y_test_untransformed = y_test_untransformed_lstm

KeyboardInterrupt: 

## 6. Diebold-Mariano Test — Pairwise Forecast Comparison

The DM test (with Harvey adjustment) compares pairwise forecast accuracy across all models for K = 6 (as reported in Table 5 of the paper). A positive DM statistic indicates the row model forecasts better than the column model. Significance: * 10%, ** 5%, *** 1%.

In [18]:
k6_idx = 4  # K=6 is the 5th element (0-indexed) in splits [2,3,4,5,6,7]

model_names = ['AR(1)', 'RF', 'GB', 'XGBoost', 'SVR', 'LASSO', 'LSTM']
preds = {
    'AR(1)' : y_pred_untransformed_ar,
    'RF'    : y_pred_random_forest[k6_idx],
    'GB'    : y_pred_gradboost[k6_idx],
    'XGBoost': y_pred_xgboost[k6_idx],
    'SVR'   : y_pred_svr[k6_idx],
    'LASSO' : y_pred_lasso[k6_idx],
    'LSTM'  : y_pred_lstm,
}

def significance_stars(p):
    if p < 0.01:  return '***'
    elif p < 0.05: return '**'
    elif p < 0.10: return '*'
    else:          return ''

print("Diebold-Mariano Test Matrix (K=6, Harvey adjustment, h=1)")
print("Rows = pred1, Cols = pred2")
print()

header = f"{'':<10}" + "".join(f"{n:>14}" for n in model_names)
print(header)
print("-" * len(header))

for row_name in model_names:
    row_str = f"{row_name:<10}"
    for col_name in model_names:
        if row_name == col_name:
            row_str += f"{'–':>14}"
        else:
            result = dm_test(y_test_untransformed,
                             preds[row_name], preds[col_name],
                             h=1, harvey_adj=True)
            stars = significance_stars(result.p_value)
            cell = f"{result.DM:.3f}{stars}"
            row_str += f"{cell:>14}"
    print(row_str)

print()
print("Note: * p<0.10, ** p<0.05, *** p<0.01")

NameError: name 'y_pred_lstm' is not defined